# **Load Bronze Tables (Import & Load)**

In [0]:
# ============================================
# CELL 1: Imports & Load Bronze Tables (Final)
# ============================================
from pyspark.sql.functions import col, to_date, year, month, when, lower, trim, current_date, count, desc
from pyspark.sql.types import DoubleType, IntegerType, StringType

# Correct mapping – jaise tune bronze tables banaye the
df_sales_raw = spark.table("workspace.default.bronze_quick")      # Sales data
df_orders_raw = spark.table("workspace.default.bronze_pricing")   # Orders data

print("Bronze tables loaded successfully")
print(f"Sales rows: {df_sales_raw.count()}, Orders rows: {df_orders_raw.count()}")

Bronze tables loaded successfully
Sales rows: 30600, Orders rows: 100000


# **Sales Table: Missing Value Handling (Fill Nulls)**

In [0]:
# ============================================
# CELL 2: Sales Table – Handle Missing Values
# ============================================
df_sales_clean = df_sales_raw.dropDuplicates(["order_id"])

# Numeric columns -> 0
numeric_cols_sales = ['base_price', 'discount_percent', 'final_price', 'units_sold', 'revenue', 'customer_age']
for c in numeric_cols_sales:
    if c in df_sales_clean.columns:
        df_sales_clean = df_sales_clean.fillna({c: 0})

# String columns -> 'unknown'
string_cols_sales = ['category', 'state', 'zone', 'brand_type', 'customer_gender', 
                     'sales_event', 'competition_intensity', 'inventory_pressure']
for c in string_cols_sales:
    if c in df_sales_clean.columns:
        df_sales_clean = df_sales_clean.fillna({c: 'unknown'})

# Date column – bronze_quick mein 'order_date' hai
date_col_sales = 'order_date'   # directly set kyunki column exist karta hai
if date_col_sales not in df_sales_clean.columns:
    raise Exception(f"Date column '{date_col_sales}' not found in sales table")

# Fill null date with mode (convert mode to string)
mode_date = df_sales_clean.filter(col(date_col_sales).isNotNull()) \
    .groupBy(date_col_sales) \
    .agg(count("*").alias("cnt")) \
    .orderBy(desc("cnt")) \
    .limit(1) \
    .collect()
if mode_date:
    mode_val = str(mode_date[0][0])
    df_sales_clean = df_sales_clean.fillna({date_col_sales: mode_val})
else:
    df_sales_clean = df_sales_clean.fillna({date_col_sales: "1900-01-01"})

print("Sales table: missing values handled")

Sales table: missing values handled


#  **Sales Table: Type Casting & Cleaning**

In [0]:
# ============================================
# CELL 3: Sales Table – Type Casting & Standardization
# ============================================
# Numeric casts
for c in ['base_price', 'discount_percent', 'final_price', 'revenue']:
    if c in df_sales_clean.columns:
        df_sales_clean = df_sales_clean.withColumn(c, col(c).cast(DoubleType()))
for c in ['units_sold', 'customer_age']:
    if c in df_sales_clean.columns:
        df_sales_clean = df_sales_clean.withColumn(c, col(c).cast(IntegerType()))

# String cleaning
string_cols = ['category', 'state', 'zone', 'brand_type', 'customer_gender', 
               'sales_event', 'competition_intensity', 'inventory_pressure']
for c in string_cols:
    if c in df_sales_clean.columns:
        df_sales_clean = df_sales_clean.withColumn(c, lower(trim(col(c))))

# Date conversion
df_sales_clean = df_sales_clean \
    .withColumn("order_date", to_date(col("order_date"), "dd-MM-yyyy")) \
    .withColumn("year", year("order_date")) \
    .withColumn("month", month("order_date"))

# Remove rows where date conversion failed
df_sales_clean = df_sales_clean.filter(col("order_date").isNotNull())

# Filter negative revenue and future dates
if 'revenue' in df_sales_clean.columns:
    df_sales_clean = df_sales_clean.filter((col("revenue") >= 0) & (col("order_date") <= current_date()))

print(f"Sales table ready: {df_sales_clean.count()} rows, {len(df_sales_clean.columns)} cols")
df_sales_clean.printSchema()

Sales table ready: 30600 rows, 21 cols
root
 |-- order_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- state: string (nullable = false)
 |-- zone: string (nullable = false)
 |-- category: string (nullable = false)
 |-- brand_type: string (nullable = false)
 |-- customer_gender: string (nullable = false)
 |-- customer_age: integer (nullable = false)
 |-- base_price: double (nullable = false)
 |-- discount_percent: double (nullable = false)
 |-- final_price: double (nullable = false)
 |-- units_sold: integer (nullable = false)
 |-- revenue: double (nullable = false)
 |-- sales_event: string (nullable = false)
 |-- competition_intensity: string (nullable = false)
 |-- inventory_pressure: string (nullable = false)
 |-- _rescued_data: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _bronze_ingested_at: timestamp (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)



# **Orders Table: Missing Value**

In [0]:
# ============================================
# CELL 4: Orders Table – Handle Missing Values
# ============================================
df_orders_clean = df_orders_raw.dropDuplicates(["order_id"])

# Numeric columns -> 0
numeric_cols_orders = ['delivery_time_min', 'order_value_inr', 'service_rating', 'product_category_id', 
                       'hour', 'order_hour']
for c in numeric_cols_orders:
    if c in df_orders_clean.columns:
        df_orders_clean = df_orders_clean.fillna({c: 0})

# String columns -> 'unknown'
string_cols_orders = ['platform_name', 'product_category_name', 'sla_delay', 'Segment', 'weekday', 
                      'customer_feedback', 'delivery_delay', 'refund_requested']
for c in string_cols_orders:
    if c in df_orders_clean.columns:
        df_orders_clean = df_orders_clean.fillna({c: 'unknown'})

# Datetime column – bronze_pricing mein 'order_datetime' hai
dt_col_orders = 'order_datetime'
if dt_col_orders not in df_orders_clean.columns:
    raise Exception(f"Datetime column '{dt_col_orders}' not found in orders table")

# Fill null datetime with mode (convert to string)
mode_dt = df_orders_clean.filter(col(dt_col_orders).isNotNull()) \
    .groupBy(dt_col_orders) \
    .agg(count("*").alias("cnt")) \
    .orderBy(desc("cnt")) \
    .limit(1) \
    .collect()
if mode_dt:
    mode_val = str(mode_dt[0][0])
    df_orders_clean = df_orders_clean.fillna({dt_col_orders: mode_val})
else:
    df_orders_clean = df_orders_clean.fillna({dt_col_orders: "1900-01-01 00:00"})

print("Orders table: missing values handled")

Orders table: missing values handled


# **Orders Table: Type Casting & Standardization**

In [0]:
# ============================================
# CELL 5: Orders Table – Type Casting & Standardization
# ============================================
# Numeric casts
if 'delivery_time_min' in df_orders_clean.columns:
    df_orders_clean = df_orders_clean.withColumn("delivery_time_min", col("delivery_time_min").cast(IntegerType()))
if 'order_value_inr' in df_orders_clean.columns:
    df_orders_clean = df_orders_clean.withColumn("order_value_inr", col("order_value_inr").cast(DoubleType()))
if 'service_rating' in df_orders_clean.columns:
    df_orders_clean = df_orders_clean.withColumn("service_rating", col("service_rating").cast(DoubleType()))
if 'product_category_id' in df_orders_clean.columns:
    df_orders_clean = df_orders_clean.withColumn("product_category_id", col("product_category_id").cast(IntegerType()))
if 'hour' in df_orders_clean.columns:
    df_orders_clean = df_orders_clean.withColumn("hour", col("hour").cast(IntegerType()))
if 'order_hour' in df_orders_clean.columns:
    df_orders_clean = df_orders_clean.withColumn("order_hour", col("order_hour").cast(IntegerType()))

# String cleaning
string_orders = ['platform_name', 'product_category_name', 'sla_delay', 'Segment', 'weekday', 
                 'customer_feedback', 'delivery_delay', 'refund_requested']
for c in string_orders:
    if c in df_orders_clean.columns:
        df_orders_clean = df_orders_clean.withColumn(c, lower(trim(col(c))))

# Datetime conversion
df_orders_clean = df_orders_clean \
    .withColumn("order_datetime", to_date(col("order_datetime"), "dd-MM-yyyy HH:mm")) \
    .withColumn("order_date", col("order_datetime")) \
    .withColumn("year", year("order_datetime")) \
    .withColumn("month", month("order_datetime"))

# Remove rows where date conversion failed
df_orders_clean = df_orders_clean.filter(col("order_datetime").isNotNull())

# Create flag columns
if 'delivery_delay' in df_orders_clean.columns:
    df_orders_clean = df_orders_clean.withColumn("delivery_delay_flag", when(col("delivery_delay") == "yes", 1).otherwise(0))
if 'refund_requested' in df_orders_clean.columns:
    df_orders_clean = df_orders_clean.withColumn("refund_flag", when(col("refund_requested") == "yes", 1).otherwise(0))

# Filter negative order value and future dates
if 'order_value_inr' in df_orders_clean.columns:
    df_orders_clean = df_orders_clean.filter((col("order_value_inr") >= 0) & (col("order_datetime") <= current_date()))

print(f"Orders table ready: {df_orders_clean.count()} rows, {len(df_orders_clean.columns)} cols")
df_orders_clean.printSchema()

Orders table ready: 100000 rows, 26 cols
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- platform_name: string (nullable = false)
 |-- product_category_id: integer (nullable = false)
 |-- order_datetime: date (nullable = false)
 |-- delivery_time_min: integer (nullable = false)
 |-- order_value_inr: double (nullable = false)
 |-- delivery_delay: string (nullable = false)
 |-- refund_requested: string (nullable = true)
 |-- service_rating: double (nullable = false)
 |-- customer_feedback: string (nullable = false)
 |-- product_category_name: string (nullable = false)
 |-- sla_delay: string (nullable = false)
 |-- Segment: string (nullable = false)
 |-- hour: integer (nullable = false)
 |-- weekday: string (nullable = false)
 |-- date: date (nullable = true)
 |-- order_hour: integer (nullable = false)
 |-- _rescued_data: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _bronze_ingested_at: timestamp (nullable = true)
 

# **Save Silver Tables**

In [0]:
# ============================================
# CELL 6: Save Cleaned Tables to Silver Layer
# ============================================
# ============================================
# CELL 6: Save Silver Tables (Overwrite – idempotent, no duplicates)
# ============================================
df_sales_clean.write.mode("overwrite").saveAsTable("silver_sales_clean")
df_orders_clean.write.mode("overwrite").saveAsTable("silver_orders_clean")

print("Silver tables saved (full overwrite – complete history from bronze)")

Silver tables saved (full overwrite – complete history from bronze)
